# 01 — Exploratory Data Analysis
Análise exploratória do dataset de churn de CS.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/train_dataset.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Info geral
df.info()
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
# Estatísticas descritivas
df.describe().round(2)

## Distribuição de Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Churn
churn_counts = df['renovado'].value_counts()
axes[0].pie(
    churn_counts.values,
    labels=['Churnou', 'Renovou'],
    colors=['#e74c3c', '#2ecc71'],
    autopct='%1.1f%%',
    startangle=90,
)
axes[0].set_title('Distribuição de Churn')

# Upsell
upsell_counts = df['fez_upsell'].value_counts()
axes[1].pie(
    upsell_counts.values,
    labels=['Sem Upsell', 'Fez Upsell'],
    colors=['#bdc3c7', '#3498db'],
    autopct='%1.1f%%',
    startangle=90,
)
axes[1].set_title('Distribuição de Upsell')

plt.tight_layout()
plt.show()

## Features vs Churn

In [ ]:
numeric_features = [
    'engajamento_pct', 'nps_score', 'tickets_abertos',
    'dias_no_contrato', 'engagement_trend', 'nps_trend', 'mrr',
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    churned  = df[df['renovado'] == False][feat].dropna()
    renewed  = df[df['renovado'] == True][feat].dropna()
    axes[i].hist(churned, alpha=0.6, bins=25, color='#e74c3c', label='Churnou')
    axes[i].hist(renewed, alpha=0.6, bins=25, color='#2ecc71', label='Renovou')
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)

axes[-1].axis('off')  # esconde o último subplot vazio
plt.suptitle('Distribuição das Features por Label de Churn', fontsize=13)
plt.tight_layout()
plt.show()

## NPS vs Renovação por Segmento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NPS médio por renovação e segmento
pivot = df.groupby(['segment', 'renovado'])['nps_score'].mean().unstack()
pivot.columns = ['Churnou', 'Renovou']
pivot.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('NPS Médio por Segmento e Outcome')
axes[0].set_xlabel('Segmento')
axes[0].set_ylabel('NPS Médio')
axes[0].tick_params(axis='x', rotation=0)

# Taxa de churn por segmento
churn_rate = df.groupby('segment')['renovado'].apply(lambda x: 1 - x.mean())
churn_rate.plot(kind='bar', ax=axes[1], color='#e74c3c')
axes[1].set_title('Taxa de Churn por Segmento')
axes[1].set_xlabel('Segmento')
axes[1].set_ylabel('Taxa de Churn')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.show()

## Matriz de Correlação

In [ ]:
numeric_df = df[numeric_features + ['renovado', 'fez_upsell']].copy()
numeric_df['renovado']  = numeric_df['renovado'].astype(int)
numeric_df['fez_upsell'] = numeric_df['fez_upsell'].astype(int)

corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5,
)
plt.title('Correlação entre Features e Labels')
plt.tight_layout()
plt.show()

print('\nCorrelações com renovado (ordenadas):')
print(corr['renovado'].sort_values(ascending=False).to_string())

## MRR em Risco

In [ ]:
total_mrr     = df['mrr'].sum()
churned_mrr   = df[df['renovado'] == False]['mrr'].sum()
renewable_mrr = df[df['renovado'] == True]['mrr'].sum()

print(f'MRR Total        : R$ {total_mrr:,.0f}')
print(f'MRR em Risco     : R$ {churned_mrr:,.0f}  ({churned_mrr/total_mrr:.1%})')
print(f'MRR a Renovar    : R$ {renewable_mrr:,.0f} ({renewable_mrr/total_mrr:.1%})')

# Por segmento
print('\nMRR em risco por segmento:')
seg_risk = df[df['renovado'] == False].groupby('segment')['mrr'].agg(['sum','count'])
seg_risk.columns = ['MRR em Risco', 'Contas']
print(seg_risk.sort_values('MRR em Risco', ascending=False).to_string())